# Phase 3: Exploratory Data Analysis (EDA) — Telco Customer Churn

## 1. Overview & Methodological Note
This notebook evaluates feature distributions and statistical associations with customer churn.

> **Methodological Principle: Association vs. Causation**  
> Observational data establishes **statistical associations and correlations**, not causal mechanisms. In this analysis, findings are strictly framed in terms of observed relationships (e.g., *"Contract type is associated with churn"*, *"Customers with fiber optic service exhibit a higher proportion of churn"*) rather than asserting direct causality.

### Planned Plotting Matrix:
| Plot # | Plot Description | Analytical Purpose |
| :--- | :--- | :--- |
| 1 | **Churn distribution** | Target balance assessment |
| 2 | **Tenure distribution** | Customer lifetime profiling |
| 3 | **Monthly Charges distribution** | Spending behavior patterns |
| 4 | **Contract vs Churn** | Strong categorical association |
| 5 | **Payment Method vs Churn** | Payment behavior patterns |
| 6 | **Internet Service vs Churn** | Service tier relationship |
| 7 | **Senior Citizen vs Churn** | Demographic relationship |
| 8 | **Tech Support vs Churn** | Service/retention relationship |
| 9 | **Online Security vs Churn** | Service/retention relationship |
| 10 | **Total Charges vs Churn** | Cumulative spending vs retention |
| 11 | **Correlation heatmap** | Numeric collinearity & correlation with target |
| 12 | **Boxplots** | Outliers and distribution spread |

In [1]:
# 2. Environment Setup & Data Loading
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Visual style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
    'axes.labelsize': 10,
    'axes.labelweight': 'semibold'
})

# Load cleaned data
data_path = Path('../data/processed/cleaned_churn_data.csv')
df = pd.read_csv(data_path)
print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns.")

Dataset loaded: 7043 rows, 21 columns.


### Plot 1: Churn Distribution (Target Balance)
Evaluating baseline class distribution to determine imbalance.

In [2]:
counts = df['Churn'].value_counts().sort_index()
pcts = (df['Churn'].value_counts(normalize=True).sort_index() * 100).round(2)
target_df = pd.DataFrame({'Count': counts, 'Percentage (%)': pcts}, index=[0, 1])
target_df.index = ['Retained (0)', 'Churned (1)']
print("=== TARGET VARIABLE DISTRIBUTION ===")
print(target_df)

=== TARGET VARIABLE DISTRIBUTION ===
              Count  Percentage (%)
Retained (0)   5174           73.46
Churned (1)    1869           26.54


### Plot 2: Tenure Distribution (Customer Lifetime)
Analyzing customer tenure (in months) across retained and churned cohorts.

In [3]:
tenure_summary = df.groupby('Churn')['tenure'].describe().round(2)
tenure_summary.index = ['Retained (0)', 'Churned (1)']
print("=== TENURE (MONTHS) BY CHURN STATUS ===")
print(tenure_summary[['mean', 'std', 'min', '25%', '50%', '75%', 'max']])

=== TENURE (MONTHS) BY CHURN STATUS ===
               mean    std  min   25%   50%   75%   max
Retained (0)  37.57  24.11  0.0  15.0  38.0  61.0  72.0
Churned (1)   17.98  19.53  1.0   2.0  10.0  29.0  72.0


### Plot 3: Monthly Charges Distribution (Spending Behavior)
Comparing the distribution of recurring monthly expenses between customer groups.

In [4]:
monthly_summary = df.groupby('Churn')['MonthlyCharges'].describe().round(2)
monthly_summary.index = ['Retained (0)', 'Churned (1)']
print("=== MONTHLY CHARGES ($) BY CHURN STATUS ===")
print(monthly_summary[['mean', 'std', 'min', '25%', '50%', '75%', 'max']])

=== MONTHLY CHARGES ($) BY CHURN STATUS ===
               mean    std    min    25%    50%   75%     max
Retained (0)  61.27  31.09  18.25  25.10  64.43  88.4  118.75
Churned (1)   74.44  24.67  18.85  56.15  79.65  94.2  118.35


### Plot 4: Contract vs Churn (Categorical Association)
Contract length is strongly associated with churn tendency.

In [5]:
contract_agg = df.groupby('Contract')['Churn'].agg(
    Total_Customers='count',
    Churn_Count='sum',
    Churn_Rate_Pct=lambda x: (x.mean() * 100).round(2)
).loc[['Month-to-month', 'One year', 'Two year']]
print("=== CONTRACT VS CHURN ===")
print(contract_agg)

=== CONTRACT VS CHURN ===
                Total_Customers  Churn_Count  Churn_Rate_Pct
Contract                                                    
Month-to-month             3875         1655           42.71
One year                   1473          166           11.27
Two year                   1695           48            2.83


### Plot 5: Payment Method vs Churn (Payment Behavior)
Examining differences in churn proportion across payment channels.

In [6]:
payment_agg = df.groupby('PaymentMethod')['Churn'].agg(
    Total_Customers='count',
    Churn_Count='sum',
    Churn_Rate_Pct=lambda x: (x.mean() * 100).round(2)
).sort_values('Churn_Rate_Pct', ascending=False)
print("=== PAYMENT METHOD VS CHURN ===")
print(payment_agg)

=== PAYMENT METHOD VS CHURN ===
                           Total_Customers  Churn_Count  Churn_Rate_Pct
PaymentMethod                                                          
Electronic check                      2365         1071           45.29
Mailed check                          1612          308           19.11
Bank transfer (automatic)             1544          258           16.71
Credit card (automatic)               1522          232           15.24


### Plot 6: Internet Service vs Churn (Service Relationship)
Subscribers with fiber optic connections exhibit a markedly higher proportion of churn.

In [7]:
internet_agg = df.groupby('InternetService')['Churn'].agg(
    Total_Customers='count',
    Churn_Count='sum',
    Churn_Rate_Pct=lambda x: (x.mean() * 100).round(2)
).loc[['Fiber optic', 'DSL', 'No']]
print("=== INTERNET SERVICE VS CHURN ===")
print(internet_agg)

=== INTERNET SERVICE VS CHURN ===
                 Total_Customers  Churn_Count  Churn_Rate_Pct
InternetService                                              
Fiber optic                 3096         1297           41.89
DSL                         2421          459           18.96
No                          1526          113            7.40


### Plot 7: Senior Citizen vs Churn (Demographic Relationship)
Senior citizen status is associated with an elevated churn rate.

In [8]:
senior_agg = df.groupby('SeniorCitizen')['Churn'].agg(
    Total_Customers='count',
    Churn_Count='sum',
    Churn_Rate_Pct=lambda x: (x.mean() * 100).round(2)
)
senior_agg.index = ['Non-Senior (0)', 'Senior Citizen (1)']
print("=== SENIOR CITIZEN STATUS VS CHURN ===")
print(senior_agg)

=== SENIOR CITIZEN STATUS VS CHURN ===
                    Total_Customers  Churn_Count  Churn_Rate_Pct
Non-Senior (0)                 5901         1393           23.61
Senior Citizen (1)             1142          476           41.68


### Plot 8 & Plot 9: Tech Support & Online Security vs Churn (Service Retention Relationships)
Customers subscribing to value-added security and support services demonstrate substantially lower churn rates.

In [9]:
tech_agg = df.groupby('TechSupport')['Churn'].agg(Count='count', Churn_Rate_Pct=lambda x: (x.mean()*100).round(2))
sec_agg = df.groupby('OnlineSecurity')['Churn'].agg(Count='count', Churn_Rate_Pct=lambda x: (x.mean()*100).round(2))
print("=== TECH SUPPORT VS CHURN ===\n", tech_agg)
print("\n=== ONLINE SECURITY VS CHURN ===\n", sec_agg)

=== TECH SUPPORT VS CHURN ===
              Count  Churn_Rate_Pct
TechSupport                       
No            4999           31.19
Yes           2044           15.17

=== ONLINE SECURITY VS CHURN ===
                 Count  Churn_Rate_Pct
OnlineSecurity                       
No               5024           31.33
Yes              2019           14.61


### Plot 10: Total Charges vs Churn (Cumulative Spending vs Retention)
Cumulative expenditure reflects tenure length and monthly recurring charges.

In [10]:
total_summary = df.groupby('Churn')['TotalCharges'].describe().round(2)
total_summary.index = ['Retained (0)', 'Churned (1)']
print("=== TOTAL CHARGES ($) BY CHURN STATUS ===")
print(total_summary[['mean', 'std', '25%', '50%', '75%']])

=== TOTAL CHARGES ($) BY CHURN STATUS ===
                 mean      std    25%      50%      75%
Retained (0)  2549.91  2329.95  572.9  1679.52  4262.85
Churned (1)   1531.80  1890.82  134.5   703.55  2331.30


### Plot 11: Correlation Heatmap (Numeric Relationships)
Evaluating linear relationships between continuous attributes, demographic indicators, and target churn.

In [11]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen', 'Churn']
corr_matrix = df[num_cols].corr().round(3)
print("=== CORRELATION MATRIX ===")
print(corr_matrix)

=== CORRELATION MATRIX ===
                tenure  MonthlyCharges  TotalCharges  SeniorCitizen  Churn
tenure           1.000           0.248         0.826          0.017 -0.352
MonthlyCharges   0.248           1.000         0.651          0.220  0.193
TotalCharges     0.826           0.651         1.000          0.103 -0.198
SeniorCitizen    0.017           0.220         0.103          1.000  0.151
Churn           -0.352           0.193        -0.198          0.151  1.000


### Plot 12: Boxplots (Outliers & Distribution Spread)
Examining quartile ranges and potential outliers in numerical variables.

In [12]:
print("=== OUTLIER & SPREAD SUMMARY (IQR METHOD) ===")
for col in ['tenure', 'MonthlyCharges', 'TotalCharges']:
    q25, q75 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q75 - q25
    lower_bound = q25 - 1.5 * iqr
    upper_bound = q75 + 1.5 * iqr
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f"{col:15} -> IQR: {iqr:.2f} | Bounds: [{lower_bound:.2f}, {upper_bound:.2f}] | Outlier count: {len(outliers)}")

=== OUTLIER & SPREAD SUMMARY (IQR METHOD) ===
tenure          -> IQR: 46.00 | Bounds: [-60.00, 124.00] | Outlier count: 0
MonthlyCharges  -> IQR: 54.35 | Bounds: [-46.02, 171.38] | Outlier count: 0
TotalCharges    -> IQR: 3388.05 | Bounds: [-4683.52, 8868.67] | Outlier count: 0


## Summary Table of Verified Associations

| Feature | Observed Pattern | Statistical Nature |
| :--- | :--- | :--- |
| **Contract** | Month-to-month contracts have 42.71% churn vs 2.83% for 2-year. | Strong negative association between contract commitment and churn. |
| **Tenure** | Median tenure is 10 months for churned vs 38 months for retained. | Strong negative association between subscription longevity and churn. |
| **Internet Service** | Fiber optic subscribers show 41.89% churn vs 18.96% for DSL. | Notable positive association between fiber subscription and churn. |
| **Payment Method** | Electronic check users show 45.29% churn vs ~15% for automatic pay. | Strong categorical association with manual payment friction. |
| **Monthly Charges** | Churned median ($79.65) is higher than retained median ($64.43). | Moderate positive correlation with churn probability. |
| **Tech/Security Add-ons** | Absence of services is associated with ~2x higher churn proportion. | Protective inverse association with churn. |
| **Senior Citizen** | Seniors show 41.68% churn vs 23.61% for non-seniors. | Moderately positive association, correlated with fiber optic choice. |
| **Numeric Outliers** | 0 statistical outliers detected across tenure, MonthlyCharges, TotalCharges via standard 1.5*IQR. | Continuous distributions are well-bounded and ready for scaling. |